# Spikes around a dark and a light escape

For one session: pick one escape in the dark and one in the light, fetch the spikes of the matched (global) units
around each, and plot a raster, the speed and the distance from the head to each object's centre over the same 10 s.

Needs, for the session:
- `scripts/pipeline/populate_ovc_behaviour.py <experiment>`: `EscapeTrial`, `LightPeriod`, `ObjectRegistration`
- step 5 (`UnitMatching`) and, to filter by label, `scripts/pipeline/populate_bombcell.py <experiment>`

Spike times and behaviour are both on the HARP clock, so they are aligned by subtracting the stimulus onset.

Standalone: tables are read by schema name (`dj.VirtualModule`) and joined in pandas, so this runs in any repo whose
environment has `datajoint` and `aeon_mecha` (imported only to register the storage codecs) and a `datajoint.json` at
its root; the AEON_OVC code is not needed.

In [ ]:
import os
from pathlib import Path

# Run from the repo root so datajoint.json is picked up
root = Path.cwd()
while not (root / "datajoint.json").exists() and root != root.parent:
    root = root.parent
os.chdir(root)
print("Working directory:", root)

import datajoint as dj
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import aeon.dj_pipeline  # noqa: F401  registers the <xarray> / <blob@dj_store> codecs

EXPERIMENT = "ProjectAeonOVC-2026-09-14_001"
SCHEMA_PREFIX = "aeon_foragingOVC_"
ESCAPE_PARAMSET = 1
WINDOW = (-2.0, 8.0)   # s relative to stimulus onset (10 s); must lie inside the trial trajectory (-5..+15 s)
LABELS = ["good"]      # bombcell labels of the units to plot; None = every matched unit (no bombcell needed)
DARK_ONSET = None      # set a stim_onset to choose the trial yourself, e.g. "2026-09-15 10:21:03.123456"
LIGHT_ONSET = None

behav = dj.VirtualModule("ovc_behav", SCHEMA_PREFIX + "ovc_behav")
ss = dj.VirtualModule("spike_sorting", SCHEMA_PREFIX + "spike_sorting")
ephys = dj.VirtualModule("ephys", SCHEMA_PREFIX + "ephys")
bombcell = dj.VirtualModule("ovc_bombcell", SCHEMA_PREFIX + "ovc_bombcell") if LABELS else None
exp = {"experiment_name": EXPERIMENT}
trial_key = {**exp, "escape_paramset_id": ESCAPE_PARAMSET}

## Escape trials

One row per sound stimulus (stats only, no escape label). `is_dark` is the light state at onset;
`shelter_*` columns describe the shelter present at onset (`time_to_enter_s` = head first inside its polygon).

In [ ]:
def fetch_escape_trials():
    """One row per stimulus: EscapeTrial stats + sound name/duration + the shelter/object present at onset,
    widened as '<identity>_<stat>' (nearest one at onset if two of the same identity were present)."""
    stats = [a for a in behav.EscapeTrial.heading.secondary_attributes if a != "trajectory"]  # skip the file fetch
    trials = pd.DataFrame((behav.EscapeTrial & trial_key).proj(*stats).to_dicts(order_by="stim_onset"))
    if trials.empty:
        return trials
    sound = pd.DataFrame((behav.SoundStimulus.Event & exp).proj("sound_name", "duration_s").to_dicts())
    trials = trials.merge(sound[["stim_onset", "sound_name", "duration_s"]], on="stim_onset", how="left")
    objs = pd.DataFrame((behav.EscapeTrial.Object & trial_key).to_dicts())
    if objs.empty:
        return trials
    objs = objs.sort_values("distance_to_edge_at_onset").drop_duplicates(["stim_onset", "identity"])
    cols = [c for c in objs.columns if c.endswith("_at_onset") or c in
            ("min_distance_to_edge", "time_to_enter_s", "object_id", "position_category")]
    wide = objs.pivot(index="stim_onset", columns="identity", values=cols)
    wide.columns = [f"{identity.lower()}_{stat}" for stat, identity in wide.columns]
    return trials.merge(wide.reset_index(), on="stim_onset", how="left")


trials = fetch_escape_trials()
show = [c for c in ["stim_onset", "sound_name", "is_dark", "in_shelter_at_onset", "baseline_speed", "peak_speed",
                    "peak_speed_latency_s", "shelter_time_to_enter_s", "shelter_distance_to_edge_at_onset"]
        if c in trials]
print(f"{len(trials)} trials: {int((trials['is_dark'] == True).sum())} dark, "
      f"{int((trials['is_dark'] == False).sum())} light")
trials[show].round(2)

## Pick one dark and one light escape

Without an escape label, this takes, per light condition, the trial where the animal was outside the shelter at
onset, entered it within the trial window, and reached the highest peak speed (falling back to the highest peak speed
if no trial entered the shelter). Set `DARK_ONSET` / `LIGHT_ONSET` above to choose trials yourself.

In [ ]:
def pick_trial(is_dark, onset=None):
    if onset is not None:
        return trials.loc[trials["stim_onset"] == pd.Timestamp(onset)].iloc[0]
    cand = trials[trials["is_dark"] == is_dark]
    if "shelter_time_to_enter_s" in cand:
        entered = cand[cand["shelter_time_to_enter_s"].notna() & (cand["in_shelter_at_onset"] != True)]
        if len(entered):
            cand = entered
    return None if cand.empty else cand.sort_values("peak_speed", ascending=False).iloc[0]


picked = {"dark": pick_trial(True, DARK_ONSET), "light": pick_trial(False, LIGHT_ONSET)}
picked = {cond: trial for cond, trial in picked.items() if trial is not None}
for cond, trial in picked.items():
    enter = trial.get("shelter_time_to_enter_s")
    print(f"{cond:5}: {trial['stim_onset']}  {trial.get('sound_name', '')}  peak {trial['peak_speed']:.0f} cm/s  "
          f"shelter entry {'-' if pd.isna(enter) else f'{enter:.2f} s'}")

## Spikes in the window

`UnitMatching.Spikes` holds each matched (global) unit's spikes per ephys chunk, each chunk counted once (taken from
the block that owns it). A row is kept if the unit's label in that block is one of `LABELS` (bombcell labels joined in pandas
via `UnitMatching.Unit`), where a global unit that is `good` in any of its blocks counts as `good` in all of them. Only the spike rows of the ephys chunks overlapping the window are fetched.

In [ ]:
# (block, shank, global unit) combinations whose unit carries one of LABELS in that block; a global unit that is
# good in any of its blocks counts as good in all of them (stored labels unchanged)
if LABELS:
    unit_map = pd.DataFrame((ss.UnitMatching.Unit & exp).proj("global_unit").to_dicts())
    labels = pd.DataFrame((bombcell.BombcellLabel.Unit & exp).proj("bombcell_label").to_dicts())
    unit_map = unit_map.merge(labels, on=[c for c in unit_map.columns if c in labels.columns], how="left")
    good = unit_map.groupby(["insertion_number", "global_unit"])["bombcell_label"].transform(lambda s: (s == "good").any())
    unit_map["effective_label"] = unit_map["bombcell_label"].mask(good, "good")
    unit_map = unit_map[unit_map["effective_label"].isin(LABELS)]
    ALLOWED = set(zip(unit_map["block_start"], unit_map["electrode_group"], unit_map["global_unit"]))
    print(f"{len(ALLOWED)} (block, unit) pairs labelled {LABELS}")


def window_spikes(onset, window=WINDOW, labels=LABELS):
    """{(electrode_group, global_unit): spike times in s from onset} for spikes inside the window."""
    onset = pd.Timestamp(onset)
    start, end = onset + pd.Timedelta(seconds=window[0]), onset + pd.Timedelta(seconds=window[1])
    chunk_starts = (
        ephys.EphysChunk & exp & f"chunk_start <= '{end}'" & f"chunk_end >= '{start}'"
    ).to_arrays("chunk_start")
    if len(chunk_starts) == 0:
        return {}
    rows = (ss.UnitMatching.Spikes & exp & [{"chunk_start": c} for c in chunk_starts]).proj("spike_times").to_dicts()
    spikes = {}
    for row in rows:
        if labels and (row["block_start"], row["electrode_group"], row["global_unit"]) not in ALLOWED:
            continue
        t = (np.asarray(row["spike_times"]) - np.datetime64(onset)) / np.timedelta64(1, "s")
        t = t[(t >= window[0]) & (t < window[1])]
        unit = (row["electrode_group"], int(row["global_unit"]))
        spikes[unit] = np.sort(np.concatenate([spikes.get(unit, np.array([])), t]))
    return spikes


spikes = {cond: window_spikes(trial["stim_onset"]) for cond, trial in picked.items()}
for cond, s in spikes.items():
    print(f"{cond:5}: {len(s)} units, {sum(len(v) for v in s.values())} spikes in the window")

## Behaviour in the same window

From the trial's stored trajectory: centroid speed and, for every shelter/object present at onset, the distance from
the head (ear midpoint) to the object's centre.

In [ ]:
objects = pd.DataFrame((behav.ObjectRegistration.Object & exp).to_dicts()).set_index("object_id")


def window_behaviour(onset, window=WINDOW):
    key = {**trial_key, "stim_onset": pd.Timestamp(onset).to_pydatetime()}
    traj = (behav.EscapeTrial & key).fetch1("trajectory")  # -pre_s..+post_s around onset (EscapeParams)
    t = traj["time_from_onset_s"].values
    if window[0] < t.min() - 0.1 or window[1] > t.max() + 0.1:
        print(f"Window {window} reaches beyond the stored trajectory ({t.min():.1f}..{t.max():.1f} s)")
    return traj.isel(time=np.flatnonzero((t >= window[0]) & (t < window[1])))


behaviour = {cond: window_behaviour(trial["stim_onset"]) for cond, trial in picked.items()}

## Raster, speed and distance to the object centre

Rows = global units grouped by shank (same order in both columns). Grey = sound playing; dashed line = head enters
the shelter.

In [ ]:
units = sorted(set().union(*[s.keys() for s in spikes.values()]))
shanks = sorted({shank for shank, _ in units})
shank_colour = {shank: f"C{i}" for i, shank in enumerate(shanks)}

fig, axes = plt.subplots(3, len(picked), figsize=(6.5 * len(picked), 9), sharex=True, sharey="row",
                         gridspec_kw={"height_ratios": [3, 1, 1.2]}, squeeze=False)
for col, (cond, trial) in enumerate(picked.items()):
    ax_raster, ax_speed, ax_dist = axes[:, col]
    duration = (pd.Timestamp(trial["stim_offset"]) - pd.Timestamp(trial["stim_onset"])).total_seconds() \
        if pd.notna(trial["stim_offset"]) else 0
    enter = trial.get("shelter_time_to_enter_s")
    for ax in axes[:, col]:
        ax.axvspan(0, duration, color="0.92", zorder=0)
        if pd.notna(enter):
            ax.axvline(enter, color="k", ls="--", lw=1)

    # raster
    ax_raster.eventplot([spikes[cond].get(u, []) for u in units], lineoffsets=np.arange(len(units)),
                        linelengths=0.8, linewidths=0.7, colors=[shank_colour[s] for s, _ in units])
    for shank in shanks:
        rows = [i for i, (s, _) in enumerate(units) if s == shank]
        ax_raster.text(WINDOW[0], np.mean(rows), f" {shank}", color=shank_colour[shank], va="center", fontsize=8)
    ax_raster.set_ylim(-1, len(units))
    ax_raster.set_title(f"{cond}: {pd.Timestamp(trial['stim_onset']):%m-%d %H:%M:%S}, {trial.get('sound_name', '')}")
    if col == 0:
        ax_raster.set_ylabel(f"global unit ({', '.join(LABELS) if LABELS else 'all'})")

    # speed
    traj = behaviour[cond]
    t = traj["time_from_onset_s"].values
    ax_speed.plot(t, traj["speed"].values, color="k", lw=1)
    if col == 0:
        ax_speed.set_ylabel("speed (cm/s)")

    # distance from the head to each object's centre
    if "object_distance_to_centre" in traj:
        for oid in traj["object_id"].values:
            obj = objects.loc[int(oid)]
            ax_dist.plot(t, traj["object_distance_to_centre"].sel(object_id=oid).values,
                         label=f"{obj['identity']} ({obj['position_category']})")
        ax_dist.legend(fontsize=8, loc="upper right")
    else:
        ax_dist.text(0.5, 0.5, "no shelter/object present", transform=ax_dist.transAxes, ha="center")
    ax_dist.set_xlabel("time from sound onset (s)")
    if col == 0:
        ax_dist.set_ylabel("distance to centre (cm)")

axes[0, 0].set_xlim(*WINDOW)
fig.tight_layout()
plt.show()

## Notes

- Spikes and behaviour share the HARP clock; no further alignment is needed.
- The distance is measured from the head (ear midpoint) to the polygon centroid; `object_distance_to_edge` in the same
  trajectory gives the signed distance to the nearest edge (< 0 inside).
- `LABELS` filters per block: a global unit appears in a trial only if it carries one of the labels in the block that
  owns those spikes. A global unit that bombcell labels `good` in any of its blocks counts as `good` in all of them. Use `LABELS = None` to plot every matched unit.
- To see all trials, loop `window_spikes` / `window_behaviour` over `trials["stim_onset"]`.